# 03 — Preprocessing

**Changes vs original:**
- Statement `Date` shifted forward by **+1 calendar day** before the `merge_asof` so that a filing
  disclosed on day T can only appear in price snapshots from day T+1 onward. This eliminates the
  same-day look-ahead where the price had already reacted to the news.
- Target clipping (`target_1m`) **removed from this notebook**. It is now done inside `04_features`
  using only training-fold statistics so future return information never contaminates the clips.

In [1]:
import pandas as pd
import numpy as np


In [2]:
prices = pd.read_csv(
    r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processedstock_prices_clean.csv"
)

statements = pd.read_csv(
    r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\statements_quarterized.csv"
)

prices["Date"]     = pd.to_datetime(prices["Date"])
statements["Date"] = pd.to_datetime(statements["Date"])

In [3]:
prices = prices.sort_values(["SecuritiesCode", "Date"])

## Profitability ratios

In [4]:
statements["roe"] = (
    statements["q_Profit"] /
    statements["Equity"]
)

statements["roa"] = (
    statements["q_Profit"] /
    statements["TotalAssets"]
)

statements["profit_margin"] = (
    statements["q_Profit"] /
    statements["q_NetSales"]
)

statements["operating_margin"] = (
    statements["q_OperatingProfit"] /
    statements["q_NetSales"]
)

statements["asset_turnover"] = (
    statements["q_NetSales"] /
    statements["TotalAssets"]
)

statements["equity_ratio"] = (
    statements["Equity"] /
    statements["TotalAssets"]
)

statements = statements.replace([np.inf, -np.inf], np.nan)

In [5]:
# ==========================================================
# Additional profitability factors
# ==========================================================

statements["operating_roa"] = (
    statements["q_OperatingProfit"] /
    statements["TotalAssets"]
)

statements["ordinary_roa"] = (
    statements["q_OrdinaryProfit"] /
    statements["TotalAssets"]
)

statements["operating_roe"] = (
    statements["q_OperatingProfit"] /
    statements["Equity"]
)

statements["ordinary_roe"] = (
    statements["q_OrdinaryProfit"] /
    statements["Equity"]
)

statements["ordinary_margin"] = (
    statements["q_OrdinaryProfit"] /
    statements["q_NetSales"]
)

statements["profit_to_assets"] = (
    statements["q_Profit"] /
    statements["TotalAssets"]
)

statements["sales_to_equity"] = (
    statements["q_NetSales"] /
    statements["Equity"]
)

statements = statements.replace([np.inf, -np.inf], np.nan)

In [6]:
###############################################################################
# TTM Profitability
###############################################################################

statements["ttm_roa"] = (
    statements["q_Profit_ttm"]
    / statements["TotalAssets"]
)

statements["ttm_roe"] = (
    statements["q_Profit_ttm"]
    / statements["Equity"]
)

statements["ttm_operating_roa"] = (
    statements["q_OperatingProfit_ttm"]
    / statements["TotalAssets"]
)

statements["ttm_operating_roe"] = (
    statements["q_OperatingProfit_ttm"]
    / statements["Equity"]
)

In [7]:
statements["ttm_profit_margin"] = (
    statements["q_Profit_ttm"]
    / statements["q_NetSales_ttm"]
)

statements["ttm_operating_margin"] = (
    statements["q_OperatingProfit_ttm"]
    / statements["q_NetSales_ttm"]
)

statements["ttm_ordinary_margin"] = (
    statements["q_OrdinaryProfit_ttm"]
    / statements["q_NetSales_ttm"]
)

In [8]:
statements["ttm_asset_turnover"] = (
    statements["q_NetSales_ttm"]
    / statements["TotalAssets"]
)

statements["ttm_sales_to_equity"] = (
    statements["q_NetSales_ttm"]
    / statements["Equity"]
)

statements["ttm_profit_to_assets"] = (
    statements["q_Profit_ttm"]
    / statements["TotalAssets"]
)

In [9]:
ttm_cols = [
    c for c in statements.columns
    if c.startswith("ttm_")
]

for c in ttm_cols:

    lo = statements[c].quantile(0.01)
    hi = statements[c].quantile(0.99)

    statements[c] = statements[c].clip(lo, hi)

In [10]:
profitability_cols = [

    "roe",
    "roa",
    "profit_margin",
    "operating_margin",
    "asset_turnover",
    "equity_ratio",

    "operating_roa",
    "ordinary_roa",

    "operating_roe",
    "ordinary_roe",

    "ordinary_margin",

    "profit_to_assets",

    "sales_to_equity",
]

for col in profitability_cols:
    lower = statements[col].quantile(0.01)
    upper = statements[col].quantile(0.99)
    statements[col] = statements[col].clip(lower=lower, upper=upper)

## Growth features

In [11]:
growth_cols = ["q_NetSales", "q_Profit", "TotalAssets", "Equity"]

for col in growth_cols:
    statements[f"{col}_yoy"] = (
        statements
        .groupby("SecuritiesCode")[col]
        .pct_change(4)
    )

growth_features = [
    "q_NetSales_yoy",
    "q_Profit_yoy",
    "TotalAssets_yoy",
    "Equity_yoy",
]

for col in growth_features:
    lower = statements[col].quantile(0.01)
    upper = statements[col].quantile(0.99)
    statements[col] = statements[col].clip(lower, upper)

## Forecast features

In [12]:
statements["forecast_operating_margin"] = (
    statements["ForecastOperatingProfit"] /
    statements["ForecastNetSales"]
)

statements["forecast_profit_margin"] = (
    statements["ForecastProfit"] /
    statements["ForecastNetSales"]
)

statements["forecast_roa"] = (
    statements["ForecastProfit"] /
    statements["TotalAssets"]
)

statements["forecast_roe"] = (
    statements["ForecastProfit"] /
    statements["Equity"]
)

statements = statements.replace([np.inf,-np.inf],np.nan)

In [13]:
statements["forecast_sales_growth"] = (
    statements["ForecastNetSales"] /
    statements["NetSales"] - 1
)

statements["forecast_profit_growth"] = (
    statements["ForecastProfit"] /
    statements["Profit"] - 1
)

statements["forecast_eps_growth"] = (
    statements["ForecastEarningsPerShare"] /
    statements["EarningsPerShare"] - 1
)

statements = statements.replace([np.inf, -np.inf], np.nan)

forecast_cols = [

    "forecast_sales_growth",
    "forecast_profit_growth",
    "forecast_eps_growth",

    "forecast_operating_margin",
    "forecast_profit_margin",

    "forecast_roa",
    "forecast_roe",
]

for col in forecast_cols:
    lower = statements[col].quantile(0.01)
    upper = statements[col].quantile(0.99)
    statements[col] = statements[col].clip(lower, upper)

In [14]:
###############################################################################
# Fundamental Acceleration Features
###############################################################################

accel_cols = [

    "roe",
    "roa",
    "profit_margin",
    "operating_margin",
    "ordinary_margin",

    "asset_turnover",
    "sales_to_equity",

    "ttm_roe",
    "ttm_roa",
    "ttm_profit_margin",
    "ttm_asset_turnover",

]

for col in accel_cols:

    statements[f"{col}_change_4q"] = (

        statements

        .groupby("SecuritiesCode")[col]

        .diff(4)

    )
for col in statements:
    if col.endswith("change_4q"):
        lower = statements[col].quantile(0.01)
        upper = statements[col].quantile(0.99)
        statements[col] = statements[col].clip(lower, upper)

## Prepare for merge

In [15]:
financial_features = [
    "Date",
    "SecuritiesCode",
    # Raw accounting values
    "NetSales",
    "Profit",
    "TotalAssets",
    "Equity",
    # Quarterized values
    "q_NetSales",
    "q_OperatingProfit",
    "q_Profit",
    # Profitability
    "roe",
    "roa",
    "profit_margin",
    "operating_margin",
    "asset_turnover",
    "equity_ratio",
    # New profitability
    "operating_roa",
    "ordinary_roa",

    "operating_roe",
    "ordinary_roe",

    "ordinary_margin",

    "profit_to_assets",

    "sales_to_equity",

    # Growth
    "q_NetSales_yoy",
    "q_Profit_yoy",
    "TotalAssets_yoy",
    "Equity_yoy",
    # Forecast
    "forecast_sales_growth",
    "forecast_profit_growth",
    "forecast_eps_growth",
    # Forecast quality

    "forecast_operating_margin",
    "forecast_profit_margin",

    "forecast_roa",
    "forecast_roe",
    
    # ---------- TTM ----------
    "q_NetSales_ttm",
    "q_OperatingProfit_ttm",
    "q_OrdinaryProfit_ttm",
    "q_Profit_ttm",

    "ttm_roa",
    "ttm_roe",

    "ttm_operating_roa",
    "ttm_operating_roe",

    "ttm_profit_margin",
    "ttm_operating_margin",
    "ttm_ordinary_margin",

    "ttm_asset_turnover",
    "ttm_sales_to_equity",
    "ttm_profit_to_assets",
]
acceleration_features = [

    "roe_change_4q",

    "roa_change_4q",

    "profit_margin_change_4q",

    "operating_margin_change_4q",

    "ordinary_margin_change_4q",

    "asset_turnover_change_4q",

    "sales_to_equity_change_4q",

    "ttm_roe_change_4q",

    "ttm_roa_change_4q",

    "ttm_profit_margin_change_4q",

    "ttm_asset_turnover_change_4q",

]

financial_features += acceleration_features
statements_small = statements[financial_features].copy()
statements_small[["NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock", "EarningsPerShare"]]= statements[["NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock", "EarningsPerShare"]]

prices["SecuritiesCode"]           = prices["SecuritiesCode"].astype("int64")
statements_small["SecuritiesCode"] = statements_small["SecuritiesCode"].astype("int64")

## Disclosure embargo - shift statement Date by +1 day

In the raw data `Date == DisclosedDate` (verified: 100% match). This means a filing on
day T and a price snapshot also on day T would deliver same-day financial data to the model -
the price has already reacted to the news. By adding one calendar day the earliest a filing
can enter any price row is T+1, which is the first point at which an investor could
actually have traded on that information.

In [16]:
statements_small["Date"] = statements_small["Date"] + pd.Timedelta(days=1)

prices         = prices.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)
statements_small = statements_small.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

## Point-in-time merge

In [17]:
merged = pd.merge_asof(
    prices,
    statements_small,
    on="Date",
    by="SecuritiesCode",
    direction="backward",
)

print("prices :", prices.shape)
print("merged :", merged.shape)

prices : (2332531, 12)
merged : (2332531, 70)


## Resample to monthly

In [18]:
merged["YearMonth"] = merged["Date"].dt.to_period("M")

monthly = (
    merged
    .sort_values(["SecuritiesCode", "Date"])
    .groupby(["SecuritiesCode", "YearMonth"], as_index=False)
    .last()
)

print("monthly:", monthly.shape)
print("Duplicates (SecuritiesCode, YearMonth):", monthly.duplicated(["SecuritiesCode", "YearMonth"]).sum())

monthly: (116568, 71)
Duplicates (SecuritiesCode, YearMonth): 0


## Target variable — 1-month forward return

**Note:** Target clipping is intentionally omitted here. It is applied in `04_features` using
only the training fold's statistics, preventing any future return information from leaking
into the training data clips.

In [19]:
monthly["target_1m"] = (
    monthly.groupby("SecuritiesCode")["Close"]
    .shift(-1)
    / monthly["Close"]
    - 1
)

print(monthly["target_1m"].describe())

count    114554.000000
mean          0.019003
std           0.334659
min          -0.905764
25%          -0.047349
50%           0.003556
75%           0.054650
max          22.820513
Name: target_1m, dtype: float64


## Momentum features

In [20]:
for m in [1, 3, 6, 12]:
    monthly[f"mom_{m}m"] = (
        monthly.groupby("SecuritiesCode")["Close"]
        .pct_change(m)
    )

for m in [3, 6, 12]:
    ma = (
        monthly.groupby("SecuritiesCode")["Close"]
        .transform(lambda x: x.rolling(m).mean())
    )
    monthly[f"ma_{m}m_ratio"] = monthly["Close"] / ma

## Risk and liquidity features (computed on daily data)

In [21]:
merged = merged.sort_values(["SecuritiesCode", "Date"])
merged["daily_return"] = (
    merged.groupby("SecuritiesCode")["Close"].pct_change()
)
###############################################################################
# Cross-sectional market return
###############################################################################

merged["market_return"] = (
    merged.groupby("Date")["daily_return"]
          .transform("mean")
)
###############################################################################
# Residual return (beta≈1 approximation)
###############################################################################

merged["idio_return"] = (
    merged["daily_return"] - merged["market_return"]
)
###############################################################################
# Idiosyncratic volatility
###############################################################################

for days, name in zip([21, 63, 126], [1, 3, 6]):

    merged[f"idio_vol_{name}m"] = (

        merged
        .groupby("SecuritiesCode")["idio_return"]
        .transform(
            lambda x: x.rolling(days).std()
        )

    )
###############################################################################
# Idiosyncratic skewness
###############################################################################

for days, name in zip([63, 126], [3, 6]):

    merged[f"idio_skew_{name}m"] = (

        merged
        .groupby("SecuritiesCode")["idio_return"]
        .transform(
            lambda x: x.rolling(days).skew()
        )

    )

for days, name in zip([21, 63, 126, 252], [1, 3, 6, 12]):
    merged[f"vol_{name}m"] = (
        merged.groupby("SecuritiesCode")["daily_return"]
        .transform(lambda x: x.rolling(days).std())
    )

for days, name in zip([63, 126, 252], [3, 6, 12]):
    merged[f"skew_{name}m"] = (
        merged.groupby("SecuritiesCode")["daily_return"]
        .transform(lambda x: x.rolling(days).skew())
    )

for days, name in zip([63, 126, 252], [3, 6, 12]):
    merged[f"max_return_{name}m"] = (
        merged.groupby("SecuritiesCode")["daily_return"]
        .transform(lambda x: x.rolling(days).max())
    )

merged["dollar_volume"] = merged["Close"] * merged["Volume"]

for days, name in zip([21, 63], [1, 3]):
    merged[f"avg_dollar_vol_{name}m"] = (
        merged.groupby("SecuritiesCode")["dollar_volume"]
        .transform(lambda x: x.rolling(days).mean())
    )

merged["rel_volume"] = (
    merged["Volume"]
    / merged.groupby("SecuritiesCode")["Volume"]
             .transform(lambda x: x.rolling(63).mean())
)

daily_feature_cols = [
    "vol_1m", "vol_3m", "vol_6m", "vol_12m",
    "skew_3m", "skew_6m", "skew_12m",
    "max_return_3m",
    "avg_dollar_vol_1m",
    "rel_volume",
]

In [22]:
monthly = monthly.merge(
    merged[["SecuritiesCode", "Date"] + daily_feature_cols],
    on=["SecuritiesCode", "Date"],
    how="left",
)

In [23]:
# ==========================================================
# Valuation Features
# ==========================================================

EPS = 1e-8

def safe_div(num, den):
    den = den.where(den.abs() > EPS)
    return num / den

shares = monthly[
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"
]

monthly["market_cap"] = monthly["Close"] * shares

monthly["log_market_cap"] = np.log1p(monthly["market_cap"])


monthly["price_to_earnings"] = safe_div(
    monthly["Close"],
    monthly["EarningsPerShare"]
)

monthly["earnings_yield"] = safe_div(
    monthly["EarningsPerShare"],
    monthly["Close"]
)

monthly = monthly.replace([np.inf, -np.inf], np.nan)

## Feature list and global clips

In [24]:
drop_cols = ["YearMonth", "ExpectedDividend", "Target"]
monthly   = monthly.drop(columns=[c for c in drop_cols if c in monthly.columns])

# Drop highly correlated columns identified from the original notebook
monthly = monthly.drop(
    columns=[
        c for c in ["max_return_6m", "max_return_12m", "avg_dollar_vol_3m", "q_OrdinaryProfit"]
        if c in monthly.columns
    ]
)

exclude_cols = {
    "Date", "SecuritiesCode", "RowId", "target_1m",
    "Open", "High", "Low", "Close", "Volume",
    "ExpectedDividend", "Target", "market_cap"
}

feature_cols = [
    c for c in monthly.columns
    if c not in exclude_cols
    and monthly[c].dtype in ["float64", "float32", "int64", "int32"]
]

monthly = monthly.replace([np.inf, -np.inf], np.nan)

for col in feature_cols:
    lower = monthly[col].quantile(0.01)
    upper = monthly[col].quantile(0.99)
    monthly[col] = monthly[col].clip(lower, upper)

print(f"Feature count: {len(feature_cols)}")
print(monthly[feature_cols].isna().mean().sort_values(ascending=False).head(55))

Feature count: 79
ttm_profit_margin_change_4q     0.606067
ttm_asset_turnover_change_4q    0.606041
ttm_roa_change_4q               0.605861
ttm_roe_change_4q               0.605861
ttm_operating_margin            0.452912
ttm_operating_roa               0.452809
q_OperatingProfit_ttm           0.452809
ttm_operating_roe               0.452809
operating_margin_change_4q      0.436012
vol_12m                         0.430504
skew_12m                        0.430504
ttm_ordinary_margin             0.429157
q_OrdinaryProfit_ttm            0.428951
ttm_profit_margin               0.428617
ttm_sales_to_equity             0.428591
ttm_asset_turnover              0.428591
q_NetSales_ttm                  0.428591
ttm_roa                         0.428411
ttm_roe                         0.428411
ttm_profit_to_assets            0.428411
q_Profit_ttm                    0.428411
ordinary_margin_change_4q       0.410919
profit_margin_change_4q         0.410370
q_NetSales_yoy                  0.41029

## Save

In [25]:
import json

monthly.to_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\monthly_features.csv", index=False)

with open(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\processed\feature_cols.json", "w") as f:
    json.dump(feature_cols, f, indent=4)

print(f"Saved: monthly_features.csv — {monthly.shape}")
print(f"Saved: feature_cols.json   — {len(feature_cols)} features")

Saved: monthly_features.csv — (116568, 90)
Saved: feature_cols.json   — 79 features
